In [12]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.embeddings import Embeddings
from langchain_core.runnables import Runnable

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
Model = ChatGroq(model='openai/gpt-oss-20b')

In [ ]:
data = PyMuPDFLoader(r'C:\Langchain\Data\DL_UNIT_2.pdf')

In [5]:
spilitter = RecursiveCharacterTextSplitter(
    chunk_size = 80,
    chunk_overlap = 20,
    separators=['\n','\n\n'],
    keep_separator=False
)

In [8]:
chunks = spilitter.split_documents(data.load())

In [9]:
Embeddingmodel = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
)

c:\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Langchain\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nanir\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see t

In [10]:
VectorStore = Chroma(
    embedding_function = Embeddingmodel,
    collection_name = 'data',
    persist_directory = 'RAG'
)

In [11]:
VectorStore.add_documents(chunks)

['3f0fadc1-619c-4012-97a1-e5a956d0630f',
 'efad57a4-531b-4d31-a07d-4aa738a40a40',
 '3f3c06f5-38e4-42cd-a414-a750093acd5f',
 'e700915a-ebe2-42e7-a757-5d63598032d6',
 '9db46c9f-524c-42f6-8a0e-e918e42942c7',
 'a4e128cc-4f47-422f-b492-c344ebeedd2a',
 '37206724-780e-4886-99bf-db60ca900a50',
 'f6b2f007-ad74-4559-ba39-e746a8a7ebb0',
 '0f99953c-7e92-4631-8a8a-030abd8bfbcf',
 '2cff8098-013d-4c75-a9bb-b4ff4ea49eb7',
 'acf695e6-524a-4798-b43c-62372e86e4ab',
 'e1df1e9c-124f-4f41-987c-85fa1fc12e42',
 'a8e98471-3f4b-482b-b9f0-0290db34cec8',
 '2cc0054d-8893-4eb8-a277-ff8b1af0f5d6',
 '612fc4be-f0bc-47c2-b40b-1f6e484734a1',
 '0ac6a0a6-55d0-40c3-89b5-e81237052c96',
 '60c71a09-7817-4f34-afb1-6c5db16671aa',
 'b21e2d02-bdec-475f-bf38-469642581770',
 'ec4f6f26-0cdf-40a7-8aab-c9f9cffb3a7a',
 'f42d5f81-256c-4e31-bf3a-c6c3106ace05',
 'cc94f91a-4c77-474f-9606-ebceb95b7715',
 '32345d57-7ee3-4666-8cd3-96d057adf64d',
 '53dcff43-b3b1-4f5e-b29f-639d5861df35',
 '53404863-25b2-484d-ab8d-64efc039ce9d',
 '0075a525-a0bc-

In [15]:
Template = ChatPromptTemplate(
    messages=[
            ('system','You are an RAG Application  who only answers from provided documents for the user query,if provided document dont have enough context,say i dont have enough context to answer'),
            ('human','Query = {Query} , context = {docs}')
    ],
    input_variables = ['Query','docs']
)

Template

ChatPromptTemplate(input_variables=['Query', 'docs'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an RAG Application  who only answers from provided documents for the user query,if provided document dont have enough context,say i dont have enough context to answer'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['Query', 'docs'], input_types={}, partial_variables={}, template='Query = {Query} , context = {docs}'), additional_kwargs={})])

In [16]:
parser = StrOutputParser()

In [27]:
Query = 'What is Under fitting ?'

In [28]:
chain = Template | Model | parser

output = chain.invoke({'Query':Query,'docs':VectorStore.similarity_search(query=Query,k=3)})

In [29]:
output

'Under‑fitting is a situation in which a statistical model or machine‑learning algorithm is **too simple to capture the underlying patterns in the data**.  \nBecause the model cannot represent the true relationships, it performs poorly even on the training set and consequently generalises badly to new data.'